In [3]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd 
import time, threading, os, requests, ast,json

In [4]:
def scarpe(path):
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()

    try:
        auction_header = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.auction-header.row"))
        )

        auction_name = auction_header.find_element(By.TAG_NAME, "h1").text.strip()

        header_ul = auction_header.find_element(By.CSS_SELECTOR, "div.col-12.col-xl-8 ul.header-details.icon-list")
        li_items = header_ul.find_elements(By.TAG_NAME, "li")

        center = li_items[0].text.strip() if len(li_items) > 0 else ""

        time_text = li_items[1].text.strip() if len(li_items) > 1 else ""
        if " - " in time_text:
            time_part, date_part = [x.strip() for x in time_text.split(" - ")]
        else:
            time_part, date_part = time_text, ""

        data = {
            "auction_name": auction_name,
            "center": center,
            "date": date_part,
            "time": time_part
        }


        database_file = "database.json"
        if os.path.exists(database_file):
            with open(database_file, "r", encoding="utf-8") as f:
                try:
                    existing_data = json.load(f)
                    if not isinstance(existing_data, list):
                        existing_data = [existing_data]
                except:
                    existing_data = []
        else:
            existing_data = []

        existing_data.append(data)


        with open(database_file, "w", encoding="utf-8") as f:
            json.dump(existing_data, f, ensure_ascii=False, indent=4)

        print(f"✅ Auction data saved to {database_file}")

    except Exception as e:
        print(f"❌ Error scraping or saving data: {e}")
    # try:
    #     login = WebDriverWait(driver, 5).until(
    #         EC.presence_of_element_located((By.XPATH, './/a[text()="Login"]'))
    #     )
    #     login.click()
    # except:
    #     print("no login")

    # try:
    #     user_name = WebDriverWait(driver, 2).until(
    #         EC.presence_of_element_located((By.ID, 'username'))
    #     )
    #     user_name.send_keys("fourbrotherstrading@icloud.com")
    # except Exception as e:
    #     print("Username error", e)

    # try:
    #     password = WebDriverWait(driver, 2).until(
    #         EC.presence_of_element_located((By.ID, 'password'))
    #     )
    #     password.send_keys("Muhssan7865")
    # except Exception as e:
    #     print("Password error", e)

    # try:
    #     check = WebDriverWait(driver, 2).until(
    #         EC.presence_of_element_located((By.XPATH, './/button[text()="Sign in"]'))
    #     )
    #     check.click()
    # except:
    #     print("Sign-in button not found")


    car_count = 0

    try:
        if not os.path.exists("html"):
            os.makedirs("html")

        while True:  
            try:
                car_images = WebDriverWait(driver, 5).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img.card-img-top"))
                )

                for idx, img in enumerate(car_images):
                    try:
                        parent_link = img.find_element(By.XPATH, "./ancestor::a")
                        car_url = parent_link.get_attribute("href")
                        if not car_url:
                            print("No link found for this car, skipping.")
                            continue

                        driver.execute_script("window.open(arguments[0], '_blank');", car_url)
                        driver.switch_to.window(driver.window_handles[-1])
                        time.sleep(1) 
                        try:
                            li_items = WebDriverWait(driver, 5).until(
                                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.detail-item"))
                            )

                            reg_number = ""
                            for li in li_items:
                                try:
                                    span_text = li.find_element(By.TAG_NAME, "span").text.strip()
                                    if span_text.lower() == "registration": 
                                        strong_el = li.find_element(By.TAG_NAME, "strong")
                                        reg_number = strong_el.text.strip().replace("/", "_").replace(" ", "_")
                                        break
                                except Exception:
                                    continue

                            if not reg_number:
                                reg_number = f"car_{car_count+1}"

                        except Exception:
                            reg_number = f"car_{car_count+1}"


                        filename = f"html/{reg_number}.html"
                        with open(filename, "w", encoding="utf-8") as f:
                            f.write(driver.page_source)
                        print(f"✔ Saved HTML: {filename}")
                        car_count += 1

                        driver.close()
                        driver.switch_to.window(driver.window_handles[0])
                        time.sleep(1)

                    except Exception as e:
                        print("Error processing car image:", e)
                        continue

                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "li.page-item.page-item-arrow-next a")
                    next_href = next_btn.get_attribute("href")
                    if next_href:
                        print(f"➡ Moving to next page: {next_href}")
                        driver.get(next_href)
                        time.sleep(2)
                    else:
                        print("No more pages.")
                        break
                except Exception:
                    print("Pagination finished.")
                    break

            except Exception:
                print("No car images found on this page, stopping loop.")
                break

    except Exception as e:
        print("❌ Fatal error during car scraping:", e)

    print(f"\n✅ Total cars processed: {car_count}")
    
    driver.quit()

path = 'https://www.eama-norwich.co.uk/auction/100'
scarpe(path)


✅ Auction data saved to database.json
✔ Saved HTML: html/YK68WVX.html
✔ Saved HTML: html/KX64WCU.html
✔ Saved HTML: html/AK18MDO.html
✔ Saved HTML: html/YH70LYU.html
✔ Saved HTML: html/BX68BDV.html
✔ Saved HTML: html/YD68CLF.html
✔ Saved HTML: html/PN65KUF.html
✔ Saved HTML: html/OU63AKG.html
✔ Saved HTML: html/YH70LXO.html
✔ Saved HTML: html/YM16VJP.html
✔ Saved HTML: html/YK68XFE.html
✔ Saved HTML: html/DU63UWK.html
✔ Saved HTML: html/YH70LYY.html
✔ Saved HTML: html/BF19ZSW.html
✔ Saved HTML: html/YD68YXT.html
✔ Saved HTML: html/KR69MLZ.html
✔ Saved HTML: html/YH70VVJ.html
✔ Saved HTML: html/VK64OOX.html
✔ Saved HTML: html/YK68WVU.html
✔ Saved HTML: html/ND18CEN.html
✔ Saved HTML: html/YH70LYK.html
✔ Saved HTML: html/FL64XAV.html
✔ Saved HTML: html/YK68WVS.html
✔ Saved HTML: html/YH70LZB.html
✔ Saved HTML: html/DH65HRO.html
✔ Saved HTML: html/YK68XEZ.html
✔ Saved HTML: html/AE16OZR.html
✔ Saved HTML: html/YH70VUV.html
Pagination finished.

✅ Total cars processed: 28


In [ ]:
import os,re,json
import csv
from bs4 import BeautifulSoup
from datetime import datetime

with open(r"D:\bots\headers.json", "r", encoding="utf-8") as f:
    header_map = json.load(f)
headers = [header_map[k] for k in sorted(header_map, key=int)]


with open("database.json", "r", encoding="utf-8") as f:
    database = json.load(f)
if isinstance(database, list):
    auctionDetails = database[0] if database else {}

elif isinstance(database, dict):
    auction_list = database.get("auction", [])
    auctionDetails = auction_list[0] if auction_list else {}

else:
    auctionDetails = {}


def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())

    parts = folder_name.split("-")
    if not parts or not parts[0].isdigit():
        return None, None

    sheet_id = parts[0]
    name_parts = parts[1:]
    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]

    auction_name = "-".join(name_parts).strip()

    return sheet_id, auction_name

def extract_details(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    result = {}

  
    details_ul = soup.find("ul", class_="details-list")
    if details_ul:
        li_items = details_ul.find_all("li", class_="detail-item")
        for li in li_items:
            key_el = li.find("span")
            value_el = li.find("strong")
            if key_el and value_el:
                key = key_el.get_text(strip=True)
                value = value_el.get_text(strip=True)
                result[key] = value


    features_p = soup.find("p", class_="mt-4")
    if features_p:
        features_text = features_p.get_text(" ", strip=True)
        result["Features"] = features_text

    return result



def yearGeter(date_str):
    if not date_str:
        return ""

    date_str = date_str.strip().replace("\\", "").strip()

    try:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return str(dt.year)
    except:
        return ""



def extract_manual_keys():
    folder = "html"
    output_file = "eama_data.csv"

    # keys = ["Title",
    #         "Auction Name",
    #         "Sheet id",
    #         "Auction House",
    #         "Make",
    #         "Model",
    #         "Variant",
    #         "Lot", 
    #         "Year",
    #         "Reg", 
    #         "Start Time", 
    #         "Start Date",
    #         "D.O.R",
    #         "Fuel Type",
    #         "Former Keepers",
    #         "Transmission",
    #         "Colour",
    #         "MOT Expiry Date",
    #         "Keys",
    #         "Features",
    #         "VAT Status",
    #         "V5",
    #         "CAP Clean",
    #         "CAP Average",
    #         "CAP Below",
    #         "CC",
    #         "Mileage",
    #         "Mileage Warranted",
    #         "MOT Due",
    #         "Body Type",
    #         "Additional information",
    #         "General Condition",
    #         "Tyres Condition",
    #         "Inspection Report",
    #         "Images",
    #         "Damaged_images",
    #         "Damage_details",
    #         ]  

    all_rows = []

    for file in os.listdir(folder):
        if file.endswith(".html"):
            file_path = os.path.join(folder, file)
            with open(file_path, "r", encoding="utf-8") as f:
                html_content = f.read()
                soup = BeautifulSoup(html_content, "html.parser")

            row = {}

            reg_el = soup.find("span", class_="pill-item pill-item-reg")
            reg = reg_el.get_text(strip=True) if reg_el else ""
            pattern = re.compile(r'^[A-Z]{1,3}[0-9]{1,3}[A-Z]{1,3}$', re.I)
            if not pattern.match(reg):
                print(f"❌ Not valid: {reg} → Deleting file {file}")
                os.remove(file_path)
                continue
            else:
                row[header_map["5"]] = reg
                lot_el = soup.find("span", class_="pill-item pill-item-lot")
                row[header_map["9"]] = lot_el.get_text(strip=True).replace("Lot", "").strip() if lot_el else ""

                title_tag = soup.find("h1", class_="title-h1")
                row[header_map["4"]] = title_tag.get_text(strip=True) if title_tag else ""
                
                row[header_map["14"]] = auctionDetails.get("date")
                row[header_map["15"]] = auctionDetails.get("time")



                findElement = extract_details(html_content)
                mot =  findElement.get("MOT Expires", "")
                Registered =  findElement.get("Registered", "")
                row[header_map["6"]] = findElement.get("Manufacturer", "")
                row[header_map["7"]] = findElement.get("Model", "")
                row[header_map["8"]] = findElement.get("Variant", "")
                row[header_map["16"]] = Registered
                row[header_map["17"]] = yearGeter(Registered)
                row[header_map["11"]] = findElement.get("Fuel", "")
                row[header_map["25"]] = findElement.get("Former Keepers", "")
                row[header_map["12"]] = findElement.get("Transmission", "")
                row[header_map["33"]] = findElement.get("Colour", "")
                row[header_map["21"]] =mot if mot != "Expired" else " "
                row[header_map["34"]] = findElement.get("Keys", "")
                row[header_map["23"]] = findElement.get("VAT", "")
                row[header_map["20"]] = findElement.get("V5", "")
                row[header_map["35"]] = findElement.get("CAP Clean", "")
                row[header_map["36"]] = findElement.get("CAP Below", "")
                row[header_map["37"]] = findElement.get("CAP Average", "")
                row[header_map["38"]] = findElement.get("Catalogue Notes", "")
                
                cc_raw = findElement.get("CC", "")
                try:
                    cc_val = round(int(cc_raw) / 1000, 1) if cc_raw else ""
                except:
                    cc_val = ""
                row[header_map["26"]] = cc_val
                
    
                miles_raw = findElement.get("Mileage", "")
                miles_val = ""
                if miles_raw:

                    miles_clean = miles_raw.replace(",", "").replace("Miles", "").strip()

                    if miles_clean.replace(".", "").isdigit():  
                        miles_val = int(float(miles_clean)) 
                    else:
                        miles_val = ""  
                row[header_map["18"]] = miles_val
        
           
                motDue = findElement.get("Tax Due", "")
                row[header_map["21"]] = motDue if motDue != "Expired" else " "
                row[header_map["10"]] = findElement.get("Body type", "")



                row[header_map["39"]] = {}
                row[header_map["32"]] = {}
                
                
                
                base_url = "https://www.eama-norwich.co.uk/"

                inspection_div = soup.find("div", class_="header-cta")
                inspection_pdf = ""
                if inspection_div:
                    a_tag = inspection_div.find("a", href=True)
                    if a_tag:
                        href = a_tag['href'].strip()
                        if href:
                            if href.startswith("http"):
                                inspection_pdf = href
                            else:
                                inspection_pdf = base_url.rstrip("/") + "/" + href.lstrip("/")

                row[header_map["28"]] = inspection_pdf
                
                
                images_div = soup.find("div", class_="ug-thumbs-strip-inner")
                images_urls = []

                if images_div:
                    for img_tag in images_div.find_all("img", src=True):
                        src = img_tag["src"].strip()

                        if src:
                            src = src.replace(
                                "https://dgaww6lqj3.execute-api.eu-west-1.amazonaws.com/prod/buckets/eama-prod-data-s3-public/keys/",
                                "https://eama-prod-data-s3-public.s3.eu-west-1.amazonaws.com/"
                            )

                            src = src.replace("/resized/", "/")
                            src = src.split("---")[0] + ".jpg"

                            images_urls.append(src)

                row[header_map["29"]] = ", ".join(images_urls)
                damage_div = soup.find("div", class_="condition-gallery")
                damage_images = []
                damage_details = []

                if damage_div:
                    for figure in damage_div.find_all("figure", class_="ug-thumb-wrapper ug-thumb-generated ug-thumb-ratio-set"):
                        img_tag = figure.find("img", src=True)
                        if img_tag:
                            src = img_tag['src'].strip()
                            if src:
                                if src.startswith("/"):
                                    src = base_url + src
                                src = src.replace("---1140-855.jpg", "---1024-768.jpg")
                                damage_images.append(src)
                        figcaption = figure.find("figcaption")
                        if figcaption:
                            caption = " ".join(figcaption.get_text(" ", strip=True).split())
                            if caption:
                                damage_details.append(caption)

                row[header_map["30"]] = ", ".join(damage_images)
                row[header_map["31"]] = ", ".join(damage_details)
                
                Grade = ""

                grade_span = soup.find("span", class_=lambda x: x and "nama-grade-" in x)

                if grade_span:
                    classes = grade_span.get("class", [])
                    for cls in classes:
                        if cls.startswith("nama-grade-") and cls != "nama-grade":
                            Grade = cls.replace("nama-grade-", "")
                            break

                row[header_map["40"]] = Grade

                sheet_id, auction_name = get_base_folder_info()
                row[header_map["2"]] = sheet_id or ""
                row[header_map["1"]] =  auctionDetails.get("auction_name") or auction_name
                row[header_map["3"]] = "East Anglian Motor Auctions"
                row[header_map["13"]] = auctionDetails.get("center") or ""

                all_rows.append(row)
           


    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n✔ CSV Generated: {output_file}")


extract_manual_keys()



✔ CSV Generated: eama_data.csv


In [ ]:
import os
import threading
import requests
import pandas as pd
from urllib.parse import urlparse, urljoin
from PIL import Image, ImageDraw, ImageFont


def add_watermark_to_image(
    image_path,
    text="Sourced from East Anglian Motor Auctions"
):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)
        try:
            font = ImageFont.truetype("arial.ttf", 40) 
        except:
            font = ImageFont.load_default()
        padding = 20

        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]

        x = image.width - text_w - padding
        y = image.height - text_h - padding
        draw.rectangle(
            [
                x - padding,
                y - padding,
                x + text_w + padding,
                y + text_h + padding
            ],
            fill=(0, 0, 0, 180)
        )

        draw.text(
            (x, y),
            text,
            font=font,
            fill=(255, 255, 255, 255)
        )

        final = Image.alpha_composite(image, txt_layer).convert("RGB")
        final.save(image_path)

        print(f"Watermark added: {image_path}")

    except Exception as e:
        print(f"Watermark fail {image_path}: {e}")




def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg = row["Reg"]
        images = row["Images"]

        if pd.isna(images) or not str(images).strip():
            print(f"No images for {reg}")
            continue

        urls = images.split(", ")
        reg_folder = os.path.join(main_folder, reg)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, url in enumerate(urls):
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{reg}_{idx+1}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"Downloaded: {save_path}")

            except Exception as e:
                print(f"Image failed {url}: {e}")



def download_images_damage(data, main_folder="Damage_203"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg = row["Reg"]
        dmg_imgs = row["Damaged_images"]
        dmg_texts = row["Damage_details"]

        if pd.isna(dmg_imgs) or pd.isna(dmg_texts):
            print(f"No damaged images for {reg}")
            continue

        urls = dmg_imgs.split(", ")
        texts = dmg_texts.split(", ")

        reg_folder = os.path.join(main_folder, reg)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, (url, text) in enumerate(zip(urls, texts)):
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            safe_text = "".join(c if c.isalnum() or c in " _-" else "_" for c in text)

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{safe_text}_{idx+1}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"Damage downloaded: {save_path}")

            except Exception as e:
                print(f"Damage failed {url}: {e}")


df = pd.read_csv("eama_data.csv")
reg_img = df[['Reg', 'Images']]
cond_imgs = df[['Damage_details', 'Damaged_images', 'Reg']]

def start_funcs():
    threads = [
        threading.Thread(target=download_images, args=(reg_img,)),
        threading.Thread(target=download_images_damage, args=(cond_imgs,))
    ]

    for t in threads:
        t.start()

    for t in threads:
        t.join()


if __name__ == "__main__":
    start_funcs()


No damaged images for AE16OZR
No damaged images for AK18MDO
No damaged images for BF19ZSW
No damaged images for BX68BDV
No damaged images for DH65HRO
No damaged images for DU63UWK
No damaged images for FL64XAV
No damaged images for KR69MLZ
No damaged images for KX64WCU
No damaged images for ND18CEN
No damaged images for OU63AKG
No damaged images for PN65KUF
No damaged images for VK64OOX
No damaged images for YD68CLF
No damaged images for YD68YXT
No damaged images for YH70LXO
No damaged images for YH70LYK
No damaged images for YH70LYU
No damaged images for YH70LYY
No damaged images for YH70LZB
No damaged images for YH70VUV
No damaged images for YH70VVJ
No damaged images for YK68WVS
No damaged images for YK68WVU
No damaged images for YK68WVX
No damaged images for YK68XEZ
No damaged images for YK68XFE
No damaged images for YM16VJP
Watermark added: Images\AE16OZR\AE16OZR_1.jpg
Downloaded: Images\AE16OZR\AE16OZR_1.jpg
Watermark added: Images\AE16OZR\AE16OZR_2.jpg
Downloaded: Images\AE16OZR\

In [ ]:
import os
import time
import pandas as pd
import requests
import json

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


EMAIL = "fourbrotherstrading@icloud.com"
PASSWORD = "Muhssan7865"


def setup_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver



def login(driver):
    driver.get("https://www.protruckauctions.co.uk/login")
    wait = WebDriverWait(driver, 10)

    wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys(EMAIL)
    driver.find_element(By.ID, "password").send_keys(PASSWORD)
    driver.find_element(By.ID, "sign-in").click()

    time.sleep(2)
    print("[+] Login successful!")



def get_session_cookies(driver):
    cookies = driver.get_cookies()
    session = requests.Session()

    for cookie in cookies:
        session.cookies.set(cookie['name'], cookie['value'])

    return session



def download_pdf(session, url, save_path):
    print(f"[+] Downloading: {url}")

    response = session.get(url)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)

        print(f"[✔] Saved: {save_path}")
    else:
        print(f"[X] Failed ({response.status_code}) : {url}")



def download_all_pdfs(csv_file):
    df = pd.read_csv(csv_file)

    base_folder = "Inspection Reports"
    os.makedirs(base_folder, exist_ok=True)

    driver = setup_driver()
    login(driver)

    session = get_session_cookies(driver)   
    driver.quit()

    for _, row in df.iterrows():
        reg = row["Reg"]
        url = row["Inspection Report"]

        if pd.isna(url) or not str(url).strip():
            print(f"[!] Missing PDF for {reg}")
            continue

        save_path = os.path.join(base_folder, f"{reg}.pdf")
        download_pdf(session, url, save_path)

    print("\nAll PDFs downloaded!")



if __name__ == "__main__":
    download_all_pdfs("protruck_data.csv")


[+] Login successful!
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/35083.pdf
[✔] Saved: Inspection Reports\AV64RZF.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/52396.pdf
[✔] Saved: Inspection Reports\BF71XYZ.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/51902.pdf
[✔] Saved: Inspection Reports\BJ21NBM.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/51551.pdf
[✔] Saved: Inspection Reports\BJ22YON.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/49607.pdf
[✔] Saved: Inspection Reports\BK23XDF.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/52795.pdf
[✔] Saved: Inspection Reports\BL22ONO.pdf
[+] Downloading: https://www.protruckauctions.co.uk/motorvehicleinspectionreport/standard/47799.pdf
[✔] Saved: Inspection Repo

In [ ]:
import os
from PyPDF2 import PdfReader, PdfWriter, Transformation
from reportlab.pdfgen import canvas

HEADER_HEIGHT = 25  


def create_header_page(text, filename, page_width, page_height):
    c = canvas.Canvas(filename, pagesize=(page_width, page_height))


    r, g, b = (4/255, 122/255, 250/255)


    c.setFillColorRGB(r, g, b)
    c.rect(0, page_height - HEADER_HEIGHT, page_width, HEADER_HEIGHT, fill=1)


    c.setFillColorRGB(1, 1, 1)
    c.setFont("Helvetica-Bold", 12)

    text_width = c.stringWidth(text, "Helvetica-Bold", 12)
    x = (page_width - text_width) / 2
    y = page_height - HEADER_HEIGHT + 7

    c.drawString(x, y, text)
    c.save()


def add_header_to_pdf(input_pdf, output_pdf):
    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    for page in reader.pages:
        page_width = float(page.mediabox.width)
        page_height = float(page.mediabox.height)

        shift = Transformation().translate(0, -HEADER_HEIGHT)
        page.add_transformation(shift)

        temp_header = "header_temp.pdf"
        create_header_page("Source from Protruck Auctions", temp_header, page_width, page_height)

        header_pdf = PdfReader(temp_header)
        header_page = header_pdf.pages[0]

        page.merge_page(header_page)
        writer.add_page(page)

    with open(output_pdf, "wb") as f:
        writer.write(f)

    os.remove(temp_header)


def add_header_to_all_pdfs(folder):
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            input_pdf = os.path.join(folder, file)
            output_pdf = os.path.join(folder, file)

            print(f"Fixing & adding new header to: {file}")
            add_header_to_pdf(input_pdf, output_pdf)

    print("\n✔ All PDFs updated with blue header (#047AFA)!")


if __name__ == "__main__":
    add_header_to_all_pdfs("Inspection Reports")


Fixing & adding new header to: AV64RZF.pdf
Fixing & adding new header to: BF71XYZ.pdf
Fixing & adding new header to: BJ21NBM.pdf
Fixing & adding new header to: BJ22YON.pdf
Fixing & adding new header to: BK23XDF.pdf
Fixing & adding new header to: BL22ONO.pdf
Fixing & adding new header to: BT70UAS.pdf
Fixing & adding new header to: CE70LTV.pdf
Fixing & adding new header to: DE21VJL.pdf
Fixing & adding new header to: DE21VLG.pdf
Fixing & adding new header to: DX23CFL.pdf
Fixing & adding new header to: FD65ORA.pdf
Fixing & adding new header to: FG71RMZ.pdf
Fixing & adding new header to: FJ20UZW.pdf
Fixing & adding new header to: FL18ENO.pdf
Fixing & adding new header to: FL65RCO.pdf
Fixing & adding new header to: FP65YUO.pdf
Fixing & adding new header to: FY24EOS.pdf
Fixing & adding new header to: HV11FMF.pdf
Fixing & adding new header to: HV24ZYA.pdf
Fixing & adding new header to: HV74KGU.pdf
Fixing & adding new header to: HY18TSU.pdf
Fixing & adding new header to: KX13JXC.pdf
Fixing & ad